# 15 — Real-data SCC-versus-BCC specialist
**Use a T4 GPU. Run all cells in order.** Self-contained: no variables from earlier notebook sessions required.

Train a binary specialist on the real BCC and SCC training images only. Copy the backbone weights from the completed notebook 13 model, preserving that source model. Binary output: **0 = BCC, 1 = SCC**. Use inverse-frequency binary class weights once, no oversampling or synthetic images.

The combined classifier keeps every melanoma prediction made by notebook 13's unchanged three-class model. For other images, classify SCC if the specialist sigmoid output is ≥0.5; otherwise classify BCC. The 0.5 threshold is fixed before training. This routing can miss SCC cases already routed to melanoma; calculate and report its recall ceiling before fitting.

Select a trained specialist checkpoint by an SCC-focused validation objective on the **two separate, fixed, full three-class validation sets**. The objective is `4 × all_targets_met + 2 × other_class_recalls_preserved + minimum_SCC_F1`. Here all targets require accuracy ≥0.85, macro F1 >0.80 and all class recalls ≥0.80 on BOTH sets; preservation requires BCC and melanoma recall ≥0.80 on both. Compare the selected specialist with the unchanged baseline using that same objective. If it does not improve the objective, retain the baseline as the recommendation.

This is a bounded real-data baseline for a later synthetic-data comparison, not a guarantee of target attainment. No test image data or predictions are loaded, and no Streamlit package or existing model is replaced. The original test set has been inspected previously; do not call it a fresh final test. Patient independence and darker-skin reliability remain unestablished.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime,timezone
from PIL import Image,ImageOps
import gc,hashlib,io,json,time
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

ROOT=Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
SOURCE_RUN='exp08_572b2a3e5ca2'
SOURCE_MODELS=ROOT/'Models'/'Transfer_Learning'/SOURCE_RUN/'ResNet50'
SOURCE_REPORTS=ROOT/'Reports'/'Training'/SOURCE_RUN/'ResNet50'
MILK_ROOT=ROOT/'Datasets'/'MILK10k'/'Prepared'/'20261104_175741_606266'
F17_PATH=ROOT/'Reports'/'External_Data_Preparation'/'external_training_manifest.csv'
CLASS_NAMES=['BCC','Melanoma','SCC'];BINARY_CLASS_NAMES=['BCC','SCC']
SEED=42;BATCH_SIZE=16;IMAGE_SIZE=(224,224)
HEAD_EPOCHS=6;FINE_EPOCHS=18;HEAD_LR=1e-3;FINE_LR=1e-5
THRESHOLD=.5
print('TensorFlow:',tf.__version__,'Keras:',getattr(tf.keras,'__version__','bundled'))
gpus=tf.config.list_physical_devices('GPU');print('Available GPUs:',gpus)
if not gpus:raise RuntimeError('Enable T4 GPU and rerun from the top.')
for gpu in gpus:
    try:tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError:pass
tf.keras.utils.set_random_seed(SEED)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def write_json(path,value):
    path=Path(path);temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2),encoding='utf-8');temporary.replace(path)
def validate_manifest(frame,split,name):
    required={'dataset','image_id','image_path','image_hash','class_label','label_index','split'}
    if not required.issubset(frame):raise ValueError(name+': missing columns')
    if frame.empty or not frame.split.eq(split).all():raise ValueError(name+': empty or wrong split')
    if not frame.class_label.map({c:i for i,c in enumerate(CLASS_NAMES)}).eq(frame.label_index).all():raise ValueError(name+': class encoding mismatch')
    if frame.duplicated(['dataset','image_id']).any() or frame.image_path.isna().any():raise ValueError(name+': duplicate IDs or missing paths')
    return frame.copy()

source_metadata=json.loads((SOURCE_MODELS/'metadata.json').read_text())
if not source_metadata.get('training_complete') or source_metadata['run_id']!=SOURCE_RUN:raise ValueError('Expected completed notebook 13 model.')
if source_metadata['class_names']!=CLASS_NAMES or source_metadata['preprocessing']!='resnet50' or source_metadata['image_size']!=list(IMAGE_SIZE):raise ValueError('Wrong source class order, preprocessing or shape.')
SOURCE_MODEL=SOURCE_MODELS/'best.keras'
if sha256_file(SOURCE_MODEL)!=source_metadata['model_sha256']:raise ValueError('Source model checksum mismatch.')
settings13=source_metadata['experiment_settings']
for split in ['train','validation','test']:
    if sha256_file(ROOT/'Datasets'/'Splits'/f'{split}_metadata.csv')!=settings13['original_split_sha256'][split]:raise ValueError('Original split changed: '+split)
    if sha256_file(MILK_ROOT/f'{split}_metadata.csv')!=settings13['milk_split_sha256'][split]:raise ValueError('MILK split changed: '+split)
if sha256_file(F17_PATH)!=settings13['external_training_manifest_sha256']:raise ValueError('Fitzpatrick training manifest changed.')
train_parts=[pd.read_csv(ROOT/'Datasets'/'Splits'/'train_metadata.csv'),pd.read_csv(F17_PATH),pd.read_csv(MILK_ROOT/'train_metadata.csv')]
full_train=validate_manifest(pd.concat(train_parts,ignore_index=True),'train','combined training')
train_frame=full_train.loc[full_train.class_label.isin(BINARY_CLASS_NAMES)].copy()
train_frame['binary_label']=train_frame.class_label.eq('SCC').astype(int)
if set(train_frame.binary_label)!={0,1}:raise ValueError('Both binary training classes are required.')
val_frames={'original':validate_manifest(pd.read_csv(ROOT/'Datasets'/'Splits'/'validation_metadata.csv'),'validation','original validation'),
            'MILK10k':validate_manifest(pd.read_csv(MILK_ROOT/'validation_metadata.csv'),'validation','MILK validation')}
fit_keys=set(zip(full_train.dataset,full_train.image_id))
for name,frame in val_frames.items():
    if set(zip(frame.dataset,frame.image_id))&fit_keys or set(frame.image_path)&set(full_train.image_path):raise ValueError('Training/validation overlap: '+name)
source_train=validate_manifest(pd.read_csv(SOURCE_REPORTS/'training_manifest.csv'),'train','source training')
cols=['dataset','image_id','class_label','image_hash','split']
a=full_train[cols].sort_values(['dataset','image_id']).reset_index(drop=True)
b=source_train[cols].sort_values(['dataset','image_id']).reset_index(drop=True)
pd.testing.assert_frame_equal(a,b,check_dtype=False)
counts=train_frame.binary_label.value_counts().reindex([0,1])
CLASS_WEIGHTS={i:float(len(train_frame)/(2*counts[i])) for i in [0,1]}
SETTINGS={'experiment':'real_SCC_BCC_specialist','source_run':SOURCE_RUN,'source_model_sha256':source_metadata['model_sha256'],
          'source_metadata_sha256':sha256_file(SOURCE_MODELS/'metadata.json'),
          'source_settings':settings13,'seed':SEED,'image_size':list(IMAGE_SIZE),'batch_size':BATCH_SIZE,
          'head_epochs':HEAD_EPOCHS,'fine_epochs':FINE_EPOCHS,'head_lr':HEAD_LR,'fine_lr':FINE_LR,
          'binary_class_order':BINARY_CLASS_NAMES,'class_weights':CLASS_WEIGHTS,'threshold':THRESHOLD,
          'routing':'preserve baseline argmax Melanoma; otherwise sigmoid>=0.5 => SCC, else BCC',
          'selection':'4*both_sets_all_targets + 2*both_sets_other_recall_preserved + min_SCC_F1',
          'synthetic_data':False,'test_inference':False,'tensorflow_version':tf.__version__,
          'keras_version':getattr(tf.keras,'__version__',None)}
RUN_ID='sccspec_'+hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:12]
MODEL_ROOT=ROOT/'Models'/'Specialists'/RUN_ID
REPORT_ROOT=ROOT/'Reports'/'Specialist_Training'/RUN_ID
LOCAL_ROOT=Path('/content/skin_cancer_specialist_images')
for p in [MODEL_ROOT,REPORT_ROOT,LOCAL_ROOT]:p.mkdir(parents=True,exist_ok=True)
if (MODEL_ROOT/'metadata.json').is_file() and json.loads((MODEL_ROOT/'metadata.json').read_text()).get('training_complete'):
    raise RuntimeError('This experiment is already complete. Read saved reports instead of retraining: '+str(REPORT_ROOT))
if (MODEL_ROOT/'best.keras').exists() or (REPORT_ROOT/'training_history.csv').exists():
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    MODEL_ROOT.rename(MODEL_ROOT.with_name(RUN_ID+'_interrupted_'+stamp));MODEL_ROOT.mkdir()
    REPORT_ROOT.rename(REPORT_ROOT.with_name(RUN_ID+'_interrupted_'+stamp));REPORT_ROOT.mkdir()
write_json(REPORT_ROOT/'training_settings.json',SETTINGS)
train_frame.to_csv(REPORT_ROOT/'binary_training_manifest.csv',index=False)
print('Specialist run:',RUN_ID,'Training images:',len(train_frame))
display(pd.crosstab(train_frame.dataset,train_frame.class_label))
print('Binary class weights:',CLASS_WEIGHTS)
print('No test images or predictions loaded.')

## Verify the baseline and routing ceiling
Align stored baseline validation scores with the current manifests by source and image ID. Verify that their metrics reproduce the completed model's recorded results. Calculate the maximum SCC recall possible under the fixed melanoma gate; stop before training if that gate alone makes 80% SCC recall unreachable on either validation set.

In [ ]:
def class_metrics(labels,predicted):
    labels=np.asarray(labels,dtype=np.int64);predicted=np.asarray(predicted,dtype=np.int64)
    if labels.shape!=predicted.shape or labels.ndim!=1 or not len(labels):raise ValueError('Invalid classification labels.')
    if not np.isin(labels,[0,1,2]).all() or not np.isin(predicted,[0,1,2]).all():raise ValueError('Invalid class indices.')
    matrix=np.zeros((3,3),dtype=np.int64);np.add.at(matrix,(labels,predicted),1)
    support=matrix.sum(axis=1)
    if (support==0).any():raise ValueError('All three validation classes are required.')
    tp=matrix.diagonal().astype(float)
    precision=np.divide(tp,matrix.sum(axis=0),out=np.zeros(3),where=matrix.sum(axis=0)>0)
    recall=tp/support
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(3),where=(precision+recall)>0)
    summary={'accuracy':float(tp.sum()/len(labels)),'macro_f1':float(f1.mean()),'BCC_recall':float(recall[0]),
             'Melanoma_recall':float(recall[1]),'SCC_precision':float(precision[2]),'SCC_recall':float(recall[2]),'SCC_f1':float(f1[2])}
    table=pd.DataFrame({'class_label':CLASS_NAMES,'precision':precision,'recall':recall,'f1':f1,'support':support,
                        'true_positive':tp.astype(int),'false_negative':support-tp.astype(int)})
    return summary,table,matrix

def combine_predictions(base_scores,scc_scores,threshold=.5):
    base_scores=np.asarray(base_scores,dtype=float);scc_scores=np.asarray(scc_scores,dtype=float).reshape(-1)
    if base_scores.shape!=(len(scc_scores),3) or not np.isfinite(base_scores).all() or not np.isfinite(scc_scores).all():raise ValueError('Invalid classifier score shapes/values.')
    if (base_scores<0).any() or (base_scores>1).any() or not np.allclose(base_scores.sum(axis=1),1,atol=1e-4):raise ValueError('Invalid baseline softmax scores.')
    if (scc_scores<0).any() or (scc_scores>1).any() or not 0<threshold<1:raise ValueError('Invalid specialist sigmoid scores or threshold.')
    return np.where(base_scores.argmax(axis=1)==1,1,np.where(scc_scores>=threshold,2,0)).astype(np.int64)

def validation_objective(results):
    passed=all(v['accuracy']>=.85 and v['macro_f1']>.80 and min(v['BCC_recall'],v['Melanoma_recall'],v['SCC_recall'])>=.80 for v in results.values())
    preserves=all(min(v['BCC_recall'],v['Melanoma_recall'])>=.80 for v in results.values())
    return float(4*passed+2*preserves+min(v['SCC_f1'] for v in results.values())),bool(passed),bool(preserves)

BASE_SCORES={};VAL_LABELS={};BASE_RESULTS={};prediction_hashes={};ceiling_rows=[]
for name,frame in val_frames.items():
    p=SOURCE_REPORTS/f'best_{name}_validation_predictions.csv'
    saved=pd.read_csv(p)
    if saved.duplicated(['dataset','image_id']).any() or len(saved)!=len(frame) or not saved.split.eq('validation').all():raise ValueError('Wrong baseline predictions: '+name)
    aligned=frame[['dataset','image_id','class_label']].merge(saved,on=['dataset','image_id'],how='left',suffixes=('_manifest',''),validate='one_to_one')
    if not aligned.class_label_manifest.eq(aligned.class_label).all():raise ValueError('Baseline prediction labels/IDs mismatch: '+name)
    probs=aligned[[f'score_{c}' for c in CLASS_NAMES]].to_numpy(dtype=float)
    # Validate probabilities without changing baseline predictions.
    combine_predictions(probs,np.zeros(len(probs)),THRESHOLD)
    labels=frame.label_index.to_numpy(dtype=np.int64)
    summary,table,_=class_metrics(labels,probs.argmax(axis=1))
    for key in ['accuracy','macro_f1','SCC_precision','SCC_recall','SCC_f1']:
        np.testing.assert_allclose(summary[key],source_metadata['validation_results_by_source'][name][key],rtol=0,atol=1e-7)
    BASE_SCORES[name]=probs;VAL_LABELS[name]=labels;BASE_RESULTS[name]=summary;prediction_hashes[name]=sha256_file(p)
    true_scc=labels==2;blocked=int((true_scc&(probs.argmax(axis=1)==1)).sum());n_scc=int(true_scc.sum())
    ceiling_rows.append({'validation_set':name,'SCC_images':n_scc,'SCC_blocked_by_melanoma_gate':blocked,'maximum_possible_SCC_recall':float((n_scc-blocked)/n_scc)})
    table.to_csv(REPORT_ROOT/f'baseline_{name}_per_class.csv',index=False)
ceiling=pd.DataFrame(ceiling_rows);ceiling.to_csv(REPORT_ROOT/'routing_recall_ceiling.csv',index=False)
display(ceiling)
if ceiling.maximum_possible_SCC_recall.lt(.80).any():
    raise RuntimeError('The fixed melanoma gate prevents 80% SCC recall on at least one validation set. Do not spend GPU time on this routing design; see routing_recall_ceiling.csv.')
BASE_OBJECTIVE,BASE_PASSES,_=validation_objective(BASE_RESULTS)
write_json(REPORT_ROOT/'baseline_validation.json',BASE_RESULTS)
print('Baseline validation references passed. Routing can theoretically reach the recall target; achievement still needs training evidence.')

## Prepare local images and input pipelines
Verify original notebook 03 EXIF-corrected pixel hashes and external notebook 09/12 byte hashes. Model inputs remain original RGB orientation, resized bilinearly to 224×224 and preprocessed with ResNet50. No test pipeline is created. The Keras binary validation-loss set excludes melanoma; combined three-class validation diagnostics include every original/MILK validation image.

In [ ]:
def reference_digest(raw,dataset):
    if dataset in {'Fitzpatrick17k','MILK10k'}:return hashlib.sha256(raw).hexdigest()
    if dataset not in {'HAM10000','PAD-UFES-20'}:raise ValueError('Unknown hash scheme: '+str(dataset))
    with Image.open(io.BytesIO(raw)) as image:
        rgb=ImageOps.exif_transpose(image).convert('RGB');h=hashlib.sha256()
        h.update(f'{rgb.width}x{rgb.height}:RGB:'.encode());h.update(rgb.tobytes())
    return h.hexdigest()
for name,frame in [('binary_train',train_frame),*val_frames.items()]:
    paths=[]
    for n,row in enumerate(frame.to_dict('records'),1):
        path=Path(row['image_path'])
        if not path.is_file():raise FileNotFoundError(path)
        raw=path.read_bytes()
        if reference_digest(raw,row['dataset'])!=str(row['image_hash']):raise ValueError('Reference integrity mismatch: '+str(path))
        with Image.open(io.BytesIO(raw)) as image:
            if image.width*image.height>20_000_000:raise ValueError('Image pixel limit exceeded.')
        if path.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}:raise ValueError('Unsupported image format.')
        digest=hashlib.sha256(raw).hexdigest();target=LOCAL_ROOT/(digest+path.suffix.lower())
        if target.is_file():
            if sha256_file(target)!=digest:raise ValueError('Local cache hash mismatch.')
        else:
            temporary=target.with_suffix(target.suffix+'.part');temporary.write_bytes(raw);temporary.replace(target)
        paths.append(str(target))
        if n%500==0:print(name,n,'/',len(frame),'verified and copied',flush=True)
    frame['local_path']=paths
    print(name,'local images ready:',len(paths))

def image_dataset(frame,training=False,predict_only=False):
    labels=frame.class_label.eq('SCC').to_numpy(dtype=np.float32)
    ds=tf.data.Dataset.from_tensor_slices((frame.local_path.to_numpy(dtype=str),labels))
    if training:ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True)
    def decode(path,label):
        image=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False);image.set_shape([None,None,3])
        return tf.image.resize(tf.cast(image,tf.float32),IMAGE_SIZE,method='bilinear'),label
    ds=ds.map(decode,num_parallel_calls=4).batch(BATCH_SIZE)
    if training:
        augmentation=tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal_and_vertical',seed=SEED),
            tf.keras.layers.RandomRotation(.05,fill_mode='reflect',seed=SEED+1),
            tf.keras.layers.RandomZoom(.1,fill_mode='reflect',seed=SEED+2),
            tf.keras.layers.RandomBrightness(.1,value_range=(0,255),seed=SEED+3)])
        def prepare(images,labels):
            images=tf.clip_by_value(augmentation(images,training=True),0.,255.)
            return tf.keras.applications.resnet50.preprocess_input(images),tf.reshape(labels,(-1,1))
    else:
        def prepare(images,labels):
            images=tf.keras.applications.resnet50.preprocess_input(images)
            return images if predict_only else (images,tf.reshape(labels,(-1,1)))
    return ds.map(prepare,num_parallel_calls=1,deterministic=True).prefetch(2)
TRAIN_DS=image_dataset(train_frame,training=True)
PREDICT_DS={name:image_dataset(frame,predict_only=True) for name,frame in val_frames.items()}
binary_validation=pd.concat([frame.loc[frame.class_label.isin(BINARY_CLASS_NAMES)] for frame in val_frames.values()],ignore_index=True)
BINARY_VAL_DS=image_dataset(binary_validation)
print('Binary training and full three-class validation pipelines ready.')

## Build a separate binary model
Clone the source backbone and copy its weights; no shared trainable variables with the original three-class model. Train a new 128-unit head, then fine-tune the upper half of the backbone with batch-normalisation frozen. Binary outputs are uncalibrated conditional SCC scores, not a three-class probability distribution.

In [ ]:
def build_specialist(source_model,seed=42):
    candidates=[layer for layer in source_model.layers if isinstance(layer,tf.keras.Model) and layer.name.lower().startswith('resnet50')]
    if len(candidates)!=1:raise ValueError('Expected one nested ResNet50 backbone.')
    source_base=candidates[0]
    backbone=tf.keras.models.clone_model(source_base)
    backbone.set_weights(source_base.get_weights())
    backbone.trainable=False
    inputs=tf.keras.Input(shape=source_model.input_shape[1:],name='preprocessed_resnet50_image')
    x=backbone(inputs,training=False)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.Dense(128,activation='relu',name='binary_research_head')(x)
    x=tf.keras.layers.Dropout(.3,seed=seed)(x)
    outputs=tf.keras.layers.Dense(1,activation='sigmoid',name='conditional_SCC_score')(x)
    return tf.keras.Model(inputs,outputs,name='BCC_SCC_specialist'),backbone

tf.keras.backend.clear_session();gc.collect();tf.keras.utils.set_random_seed(SEED)
source_model=tf.keras.models.load_model(SOURCE_MODEL,compile=False,safe_mode=True)
if source_model.input_shape[1:]!=(*IMAGE_SIZE,3) or source_model.output_shape[-1]!=3:raise ValueError('Unexpected source model shape.')
# Verify one live validation batch against stored source scores before cloning.
reference_input=next(iter(PREDICT_DS['original']))
reference_base=source_model.predict(reference_input,verbose=0)
np.testing.assert_allclose(reference_base,BASE_SCORES['original'][:len(reference_base)],rtol=1e-3,atol=1e-4)
model,backbone=build_specialist(source_model,SEED)
del source_model;gc.collect()
print('Source validation preprocessing/serialization reference: passed.')
print('Binary model output:',model.output_shape,'0 = BCC; 1 = SCC')

def combined_validation(binary_model,tag=None):
    results={};binary_scores={}
    for name,ds in PREDICT_DS.items():
        conditional=binary_model.predict(ds,verbose=0).reshape(-1)
        predicted=combine_predictions(BASE_SCORES[name],conditional,THRESHOLD)
        summary,table,matrix=class_metrics(VAL_LABELS[name],predicted)
        results[name]=summary;binary_scores[name]=conditional
        if tag:
            table.to_csv(REPORT_ROOT/f'{tag}_{name}_per_class.csv',index=False)
            pd.DataFrame(matrix,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(REPORT_ROOT/f'{tag}_{name}_confusion_matrix.csv')
            out=val_frames[name].drop(columns=['local_path'],errors='ignore').copy()
            out['baseline_class']=[CLASS_NAMES[i] for i in BASE_SCORES[name].argmax(axis=1)]
            out['combined_class']=[CLASS_NAMES[i] for i in predicted]
            out['specialist_conditional_SCC_score']=conditional
            out.to_csv(REPORT_ROOT/f'{tag}_{name}_predictions.csv',index=False)
            out.loc[out.class_label!=out.combined_class].to_csv(REPORT_ROOT/f'{tag}_{name}_errors.csv',index=False)
            print(tag,name,summary);display(table)
    return results,binary_scores

class SpecialistCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__();self.best=-float('inf');self.phase='head';self.best_phase=None;self.best_epoch=None;self.rows=[]
    def on_epoch_end(self,epoch,logs=None):
        logs=logs if logs is not None else {}
        results,_=combined_validation(self.model)
        objective,passed,preserves=validation_objective(results)
        logs['val_specialist_objective']=objective
        row={'phase':self.phase,'epoch':epoch+1,'validation_targets_met':passed,'other_class_recalls_preserved':preserves,
             **{key:float(value) for key,value in logs.items()}}
        for name,metrics in results.items():row.update({name+'_'+k:v for k,v in metrics.items()})
        self.rows.append(row);pd.DataFrame(self.rows).to_csv(REPORT_ROOT/'training_history.csv',index=False)
        print(f" — original SCC recall/F1: {results['original']['SCC_recall']:.4f}/{results['original']['SCC_f1']:.4f}; MILK SCC recall/F1: {results['MILK10k']['SCC_recall']:.4f}/{results['MILK10k']['SCC_f1']:.4f}; all validation targets: {passed}")
        if objective>self.best+1e-9:
            self.best=objective;self.best_phase=self.phase;self.best_epoch=epoch+1
            pending=MODEL_ROOT/'checkpoint_pending.keras';self.model.save(pending);pending.replace(MODEL_ROOT/'best.keras')
            self.model.save_weights(MODEL_ROOT/'best.weights.h5')
            write_json(MODEL_ROOT/'checkpoint.json',{'phase':self.phase,'epoch':epoch+1,'objective':objective,'validation':results})
            print('Specialist checkpoint saved to Drive.')

def compile_model(lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='binary_accuracy')])
def callbacks():
    return [monitor,
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_specialist_objective',mode='max',factor=.5,patience=2,min_lr=1e-7,verbose=1),
            tf.keras.callbacks.EarlyStopping(monitor='val_specialist_objective',mode='max',patience=5,restore_best_weights=False)]
monitor=SpecialistCheckpoint();started=time.time()
compile_model(HEAD_LR)
print('TRAINING BCC/SCC SPECIALIST HEAD')
model.fit(TRAIN_DS,validation_data=BINARY_VAL_DS,epochs=HEAD_EPOCHS,class_weight=CLASS_WEIGHTS,callbacks=callbacks(),verbose=2)
model.load_weights(MODEL_ROOT/'best.weights.h5')
backbone.trainable=True;cutoff=int(len(backbone.layers)*.5)
for index,layer in enumerate(backbone.layers):layer.trainable=index>=cutoff and not isinstance(layer,tf.keras.layers.BatchNormalization)
monitor.phase='fine_tune';compile_model(FINE_LR)
print('FINE-TUNING BCC/SCC SPECIALIST')
model.fit(TRAIN_DS,validation_data=BINARY_VAL_DS,epochs=FINE_EPOCHS,class_weight=CLASS_WEIGHTS,callbacks=callbacks(),verbose=2)
training_seconds=time.time()-started

## Verify and report
Reload the selected specialist and compare its scores with the corresponding weights. Save complete real validation diagnostics and compare the combined classifier with the unchanged three-class baseline. A better SCC-focused validation objective does not prove test targets or skin-tone reliability.

In [ ]:
model.load_weights(MODEL_ROOT/'best.weights.h5')
expected=model.predict(reference_input,verbose=0)
loaded=tf.keras.models.load_model(MODEL_ROOT/'best.keras',compile=False,safe_mode=True)
actual=loaded.predict(reference_input,verbose=0)
np.testing.assert_allclose(expected,actual,rtol=1e-4,atol=1e-5)
np.savez(REPORT_ROOT/'validation_reference.npz',conditional_SCC_scores=actual,labels=VAL_LABELS['original'][:len(actual)])
val_frames['original'].head(len(actual)).drop(columns=['local_path'],errors='ignore').to_csv(REPORT_ROOT/'validation_reference_images.csv',index=False)
FINAL_RESULTS,conditional_scores=combined_validation(loaded,'best_specialist')
objective,passed,preserves=validation_objective(FINAL_RESULTS)
np.testing.assert_allclose(objective,monitor.best,rtol=0,atol=1e-7)
improved=objective>BASE_OBJECTIVE+1e-9
comparison=pd.DataFrame([{'validation_set':name,'classifier':stage,**results[name]}
                        for stage,results in [('unchanged_baseline',BASE_RESULTS),('combined_specialist',FINAL_RESULTS)] for name in val_frames])
comparison.to_csv(REPORT_ROOT/'validation_comparison.csv',index=False);display(comparison)
metadata={'task':'binary_BCC_vs_SCC_specialist','run_id':RUN_ID,'training_complete':True,
          'source_run':SOURCE_RUN,'source_three_class_model_sha256':source_metadata['model_sha256'],
          'model_sha256':sha256_file(MODEL_ROOT/'best.keras'),'binary_class_order':BINARY_CLASS_NAMES,
          'class_names':CLASS_NAMES,'image_size':list(IMAGE_SIZE),'preprocessing':'resnet50','preprocessing_location':'outside specialist',
          'decode':'TensorFlow RGB; no EXIF transform','resize':'bilinear stretch 224x224',
          'routing':SETTINGS['routing'],'threshold':THRESHOLD,'training_settings':SETTINGS,
          'baseline_prediction_sha256':prediction_hashes,'baseline_validation':BASE_RESULTS,'combined_validation':FINAL_RESULTS,
          'baseline_objective':BASE_OBJECTIVE,'specialist_objective':objective,'improved_validation_objective':improved,
          'all_validation_targets_met':passed,'other_class_recalls_preserved':preserves,
          'recommendation':'candidate_requires_test_evaluation' if passed and improved else ('experimental_only' if improved else 'retain_unchanged_baseline'),
          'best_phase':monitor.best_phase,'best_epoch':monitor.best_epoch,'training_seconds':round(training_seconds,2),
          'created_at_utc':datetime.now(timezone.utc).isoformat(),'synthetic_images_used':False,
          'test_targets_verified':False,'test_inference_performed':False,'darker_skin_reliability_established':False,
          'single_model_streamlit_package_compatible':False,'app_model_replaced':False}
write_json(MODEL_ROOT/'metadata.json',metadata)
print('REAL SCC/BCC SPECIALIST TRAINING COMPLETE')
print('Run ID:',RUN_ID)
print('Validation objective improved:',improved)
print('All targets met on BOTH validation sets:',passed)
print('Recommendation:',metadata['recommendation'])
print('Specialist:',MODEL_ROOT/'best.keras')
print('Reports:',REPORT_ROOT)
print('No test evaluation, synthetic generation or app model replacement performed.')

## Save and share actual results
Save this notebook to Drive and GitHub as `notebooks/15_scc_bcc_specialist_training.ipynb`.
Commit message: `Record real SCC BCC specialist training and combined validation results`.

Share the final comparison table, per-class tables, routing ceiling and final status. Record failures as well as improvements. Do not copy binary weights into the current single-model Streamlit directory: this specialist requires the original model and a separate routing implementation, and has not been test-evaluated.

This real-only specialist defines the baseline for a later synthetic comparison. A synthetic experiment must use the same fixed real training/validation partitions, architecture, schedule and selection rule; only its training augmentation should change. Generator source images must be training-only. Real independent darker-skin cases are needed to evaluate reliability; generated dark skin does not supply that evidence.